# 🏦 LendingClub Credit Risk & Loan Default Predictive Analytics
## Enterprise Executive Presentation: Scalable Default Forecasting & Risk-Adjusted Asset Pricing

---

### 💼 Executive Summary & Strategic Business Context
In peer-to-peer (P2P) lending and retail credit underwriting, platform profitability and portfolio health depend on accurate **Credit Default Risk Assessment** and **Risk-Adjusted Return on Capital (RAROC)** optimization. 

This analytics initiative presents an enterprise-grade machine learning framework deployed on **Apache Spark (PySpark)** to evaluate **2.2+ Million** loan applications from LendingClub's historical credit portfolio.

#### 🎯 Strategic Business Objectives:
1. **Capital Preservation & Expected Loss Optimization**: Predict borrower default and charge-off propensities prior to loan origination to mitigate bad-debt write-offs.
2. **Dynamic Risk-Based Pricing Strategy**: Segment borrowers into multi-tier risk categories (`Fully Paid` vs. `Late / Grace Period` vs. `Charged Off / Default`) to align interest rate risk premiums with predicted loss distributions.
3. **Scalable Big Data Enterprise Architecture**: Implement a high-throughput PySpark feature engineering and model benchmarking pipeline for real-time automated underwriting integration.

#### 📊 Analytical Framework & Portfolio Risk Architecture:
* **Target Categorization**: 
  * `Class 0: Fully Paid` (Performing Accounts)
  * `Class 1: Late (16–120 days) / Grace Period` (Early Warning Indicator)
  * `Class 2: Charged Off / Default` (Loss Write-Off Events)
* **Class Imbalance Resolution**: Controlled downsampling of performing loans to balance class proportions while retaining 100% of default loss events to maximize sensitivity.
* **Model Benchmarking Suite**: Comparative evaluation of **Logistic Regression** (Regulatory Compliance Scorecard), **Random Forest** (Non-linear Ensemble), and **Multilayer Perceptron** (Neural Network Representation).

---

In [1]:
import warnings; warnings.simplefilter("ignore", FutureWarning)  # Suppress deprecation warnings
from pyspark.sql.functions import col, count, isnan, regexp_extract, when # Data transformation functions
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, MinMaxScaler  # Feature engineering transformers
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, MultilayerPerceptronClassifier  # ML Classifiers
from pyspark.ml.evaluation import MulticlassClassificationEvaluator  # Model evaluation metrics
import pandas as pd, plotly.express as px

## 1. Enterprise Infrastructure & Spark Distributed Computing Setup
We initialize a high-throughput **Apache Spark Session** (`SparkSession`) configured for multi-core parallel compute (`local[*]`).

### 💡 Technical Rationale:
* **Distributed Computing at Scale**: Processing multi-gigabyte loan datasets with over 2.2 million rows requires in-memory distributed compute to optimize pipeline execution and model training speed.
* **Resource Optimization**: Allocated 16GB driver memory and 8 shuffle partitions to maximize multi-threading efficiency and eliminate memory bottlenecking.

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("LendingClub") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/22 00:02:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.sparkContext.setLogLevel("ERROR") # Disable WARN logs from Spark's underlying Java engine
spark.conf.set("spark.sql.ansi.enabled", "false") # Kaggle compatibility setting

## 2. Data Ingestion & Underwriting Feature Selection
We load the raw LendingClub dataset (`accepted_2007_to_2018Q4.csv.gz`) into PySpark, extracting key financial metrics critical for credit underwriting.

### 📋 Feature Inventory:
* **Borrower Credit Profile**: `annual_inc`, `dti` (Debt-to-Income ratio), `last_fico_range_high`, `last_fico_range_low`.
* **Loan Terms**: `loan_amnt`, `installment`, `int_rate`, `term`, `grade`.
* **Delinquency History**: `acc_now_delinq`, `delinq_amnt`, `tax_liens`, `recoveries`, `collection_recovery_fee`.
* **Categorical / Demographics**: `purpose`, `addr_state`, `home_ownership`, `verification_status`, `application_type`, `initial_list_status`.

In [4]:
cols = [
    "id", "purpose", "term", "verification_status", "acc_now_delinq",
    "addr_state", "annual_inc", "application_type", "dti", "grade",
    "home_ownership", "initial_list_status", "installment", "int_rate",
    "loan_amnt", "loan_status", "tax_liens", "delinq_amnt", "policy_code",
    "last_fico_range_high", "last_fico_range_low", "recoveries", "collection_recovery_fee"]

# Load dataset with schema inference & select columns natively
df = (
    spark.read.csv(
        "/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz",
        header=True,
        inferSchema=True,
    )
    .select(cols) # check null df.pandas_api().isnull().sum()
    .dropna()  # Clean nulls directly on read
)
# Preview using native Spark limit (avoids pandas conversion/warnings)
df.limit(5).toPandas()

,id,purpose,term,verification_status,acc_now_delinq,addr_state,annual_inc,application_type,dti,grade,...,int_rate,loan_amnt,loan_status,tax_liens,delinq_amnt,policy_code,last_fico_range_high,last_fico_range_low,recoveries,collection_recovery_fee
0,68407277,debt_consolidation,36 months,Not Verified,0.0,PA,55000.0,Individual,5.91,C,...,13.99,3600.0,Fully Paid,0.0,0.0,1.0,564.0,560.0,0.0,0.0
1,68355089,small_business,36 months,Not Verified,0.0,SD,65000.0,Individual,16.06,C,...,11.99,24700.0,Fully Paid,0.0,0.0,1.0,699.0,695.0,0.0,0.0
2,68341763,home_improvement,60 months,Not Verified,0.0,IL,63000.0,Joint App,10.78,B,...,10.78,20000.0,Fully Paid,0.0,0.0,1.0,704.0,700.0,0.0,0.0
3,66310712,debt_consolidation,60 months,Source Verified,0.0,NJ,110000.0,Individual,17.06,C,...,14.85,35000.0,Current,0.0,0.0,1.0,679.0,675.0,0.0,0.0
4,68476807,major_purchase,60 months,Source Verified,0.0,PA,104433.0,Individual,25.37,F,...,22.45,10400.0,Fully Paid,0.0,0.0,1.0,704.0,700.0,0.0,0.0


# 3. Exploratory Data Analysis & Feature Engineering

---

## 3.1 Categorical Consolidation: Loan Purpose (`purpose`)
Borrower loan requests span diverse categories (e.g., debt consolidation, small business, home improvement, medical expenses).

### 📈 Risk Analysis Logic:
Low-frequency categories (<300 occurrences) add noise and high dimensionality without providing statistical significance. We group sparse tail categories into a consolidated `"other"` designation to optimize model robustness and prevent overfitting.

In [5]:
df_with_count = df.groupBy("purpose").count()
df_with_count.toPandas()

,purpose,count
0,educational,404
1,home_improvement,150290
2,moving,15369
3,other,139270
4,house,14119
5,wedding,2351
6,small_business,24638
7,credit_card,516570
8,vacation,15518
9,car,23996


In [6]:
# Replacing values in the 'purpose' column based on the 'count' column condition
# If 'count' is less than 300, set 'purpose' to "other", else keep the original 'purpose'
df = df.join(df_with_count, 'purpose').withColumn("purpose", when(col("count") < 300, "other").otherwise(col("purpose"))).drop('count')

In [7]:
unique_purposes = df.select("purpose").distinct()
unique_purposes.toPandas()

,purpose
0,educational
1,home_improvement
2,moving
3,other
4,house
5,wedding
6,small_business
7,credit_card
8,vacation
9,car


---

## 3.2 Quantitative Transformation: Loan Tenure (`term`)
LendingClub issues loans under two standard duration maturities: **36 months** and **60 months**.

### 📈 Risk Analysis Logic:
60-month loans carry higher cumulative default risk due to extended macroeconomic exposure. We extract integer values (`36` and `60`) to enable numerical risk modeling.

In [8]:
df.groupby('term').count().toPandas()

,term,count
0,60 months,650191
1,36 months,1608405


In [9]:
# Extract the numeric digits and cast them to integers instantly
df = df.withColumn("term", regexp_extract(col("term"), r"\d+", 0).cast("int"))
df.groupby('term').count().toPandas()

,term,count
0,36,1608405
1,60,650191


---

## 3.3 Financial Verification Flag: Income Verification Status (`verification_status`)
Measures the depth of income documentation validated during underwriting (`Verified`, `Source Verified`, `Not Verified`).

### 📈 Risk Analysis Logic:
We engineer a binary risk indicator `verification_status_encoded`:
* **`0`**: Verified / Source Verified (High-confidence income claims)
* **`1`**: Not Verified (Self-reported income claims requiring risk adjustment)

In [10]:
df.groupby('verification_status').count().toPandas()

,verification_status,count
0,Verified,629395
1,Not Verified,743060
2,Source Verified,886141


In [11]:
# If 'verification_status' is either "Verified" or "Source Verified", set 'verification_status_encoded' to 0# Otherwise, set it to 1
# Transform and drop column in a single line
# Create the binary flag and drop the old string column instantly
df = df.withColumn("verification_status_encoded", when(col("verification_status").isin(["Verified", "Source Verified"]), 0).otherwise(1)).drop("verification_status")

In [12]:
# Verify counts using the encoded column name in a single line
df.groupBy("verification_status_encoded").count().toPandas()

,verification_status_encoded,count
0,1,743060
1,0,1515536


---

## 3.4 Outlier Management: Active Delinquencies (`acc_now_delinq`)
Reflects the count of accounts on which the borrower is currently delinquent.

### 📈 Risk Analysis Logic:
Multiple active delinquencies indicate severe credit distress. Extreme values (5 to 14 accounts) represent rare statistical outliers. We cap values at **`4`** to stabilize gradient calculations during model estimation.

In [13]:
df.groupby('acc_now_delinq').count().toPandas()

,acc_now_delinq,count
0,7.0,1
1,1.0,8290
2,5.0,3
3,0.0,2249817
4,4.0,11
5,3.0,50
6,14.0,1
7,2.0,421
8,6.0,2


In [14]:
# 1. Update the column: Convert decimals to whole integers, and cap any extreme values at 4
df = df.withColumn('acc_now_delinq', col('acc_now_delinq').cast('int')) \
       .withColumn('acc_now_delinq', when(col('acc_now_delinq') >= 4, 4).otherwise(col('acc_now_delinq'))) \
       .filter(col('acc_now_delinq').between(0, 4)) # Keep only valid records between 0 and 4

In [15]:
# 2. Shortest standard check: Display the clean distribution without needing .show()
df.groupBy('acc_now_delinq').count().toPandas()

,acc_now_delinq,count
0,1,8290
1,3,50
2,2,421
3,4,18
4,0,2249817


---

## 3.5 Applicant Structure: Individual vs. Joint Applications (`application_type`)
Evaluates credit underwriting structure across single and co-borrower loans.

### 📈 Risk Analysis Logic:
Joint applications combine income streams, reducing individual default risk. We encode Joint Applications as `0` and Individual Applications as `1`.

In [16]:
df.groupby('application_type').count().toPandas()

,application_type,count
0,Individual,2139597
1,Joint App,118999


In [17]:
# 1. Map text categories to integers (unmapped rows automatically become NULL)
df = df.withColumn('application_type', 
                   when(col('application_type') == 'Joint App', 0)
                   .when(col('application_type') == 'Individual', 1)) \
       .filter(col('application_type').isNotNull()) # Drop rows that didn't match our valid values

In [18]:
# 2. Shortest standard check: Verify the clean 0 and 1 distribution instantly
df.groupBy('application_type').count().toPandas()

,application_type,count
0,1,2139597
1,0,118999


---

## 3.6 Ordinal Indexing: Credit Rating Grade (`grade`)
LendingClub assigns internal credit grades from **A** (lowest risk) to **G** (highest risk).

### 📈 Risk Analysis Logic:
Using PySpark `StringIndexer` sorted alphabetically (`A` $
ightarrow$ `0.0`, ..., `G` $
ightarrow$ `6.0`), we preserve the ordinal risk structure to enable monotonic feature evaluation.

In [19]:
df.groupby('grade').count().toPandas()

,grade,count
0,G,12144
1,C,649471
2,B,663013
3,A,432662
4,E,135506
5,F,41758
6,D,324042


In [20]:
# Create a StringIndexer to convert 'grade' column into numerical indices
df = StringIndexer(inputCol="grade", outputCol="grade_index", stringOrderType="alphabetAsc").fit(df).transform(df).drop("grade")

In [21]:
df.groupby('grade_index').count().toPandas()

,grade_index,count
0,6.0,12144
1,1.0,663013
2,4.0,135506
3,0.0,432662
4,3.0,324042
5,2.0,649471
6,5.0,41758


---

## 3.7 Categorical Feature Encoding Pipeline (StringIndexer + OneHotEncoder)
To prepare remaining non-numeric attributes (`purpose`, `addr_state`, `home_ownership`, `initial_list_status`) for PySpark ML modeling:

1. **`StringIndexer`**: Maps string labels to numerical index values.
2. **`OneHotEncoder`**: Converts category indices into binary vector representations (`vector`) to avoid implicit distance assumptions.

In [23]:
# 1. Define column names
raw_string_cols = ["purpose", "addr_state", "home_ownership", "initial_list_status"]
index_cols = ["purpose_idx", "addr_state_idx", "home_ownership_idx", "initial_list_status_idx"]
vec_cols = ["purpose_vec", "addr_state_vec", "home_ownership_vec", "initial_list_status_vec"]

# 2. StringIndexer: Raw Strings -> _idx
indexer = StringIndexer(inputCols=raw_string_cols, outputCols=index_cols)
df = indexer.fit(df).transform(df)

# 3. OneHotEncoder: _idx -> _vec
encoder = OneHotEncoder(inputCols=index_cols, outputCols=vec_cols)
df = encoder.fit(df).transform(df)

# 4. Clean up: Drop raw strings AND intermediate _idx columns
df = df.drop(*raw_string_cols, *index_cols)

# 5. Inspect your updated schema
df.printSchema()  # Print clean schema

root
 |-- id: string (nullable = true)
 |-- term: integer (nullable = true)
 |-- acc_now_delinq: integer (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- application_type: integer (nullable = true)
 |-- dti: string (nullable = true)
 |-- installment: double (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- tax_liens: double (nullable = true)
 |-- delinq_amnt: double (nullable = true)
 |-- policy_code: string (nullable = true)
 |-- last_fico_range_high: string (nullable = true)
 |-- last_fico_range_low: string (nullable = true)
 |-- recoveries: string (nullable = true)
 |-- collection_recovery_fee: string (nullable = true)
 |-- verification_status_encoded: integer (nullable = false)
 |-- grade_index: double (nullable = false)
 |-- purpose_vec: vector (nullable = true)
 |-- addr_state_vec: vector (nullable = true)
 |-- home_ownership_vec: vector (nullable = true)
 |-- initial_

# 4. Target Definition & Portfolio Class Imbalance Resolution

---

## 4.1 Target Variable Analysis (`loan_status`)
We evaluate the distribution of historical loan performance across 9 raw status categories.

In [24]:
df.groupby('loan_status').count().toPandas()

,loan_status,count
0,Fully Paid,1076218
1,Late (31-120 days),21443
2,In Grace Period,8427
3,Late (16-30 days),4344
4,Default,40
5,Does not meet the credit policy. Status:Fully ...,1913
6,Charged Off,268452
7,Does not meet the credit policy. Status:Charge...,741
8,Current,877018


## 4.2 Multi-Class Target Encoding & Controlled Downsampling Strategy

### 🎯 Target Mapping Architecture:
* **Class 0 (`Fully Paid`)**: Performing loans with principal + interest fully recovered.
* **Class 1 (`Late 16–120 days` / `In Grace Period`)**: Early warning delinquency signals (Moderate Risk).
* **Class 2 (`Charged Off` / `Default`)**: Credit write-off events / bad debt (High Loss Risk).

### ⚖️ Class Imbalance Management Strategy:
Raw credit data exhibits heavy imbalance (~1.07M Fully Paid vs. ~268K Charged Off). 
* **Business Implication**: Training on unadjusted distributions leads models to default to majority-class predictions, missing high-cost default events.
* **Downsampling Execution**: We sample Class 0 at **30%** while retaining **100%** of Class 1 and Class 2 loss events, balancing class representation while retaining maximum default signal.

In [25]:
# 1. Encode target: 0 = Fully Paid, 1 = Late / Grace Period, 2 = Charged Off / Default
df_target = df.withColumn(
    "target",
    when(col("loan_status") == "Fully Paid", 0)
    .when(col("loan_status").isin("Late (16-30 days)", "Late (31-120 days)", "In Grace Period"), 1)
    .when(col("loan_status").isin("Charged Off", "Default"), 2)
).filter(col("target").isNotNull()).drop("loan_status")

# 2. Downsample Class 0 to 30% and save to df_downsampled
df_downsampled = df_target.sampleBy("target", fractions={0: 0.3, 1: 1.0, 2: 1.0}, seed=42)

# 3. Check downsampled counts
df_downsampled.groupBy("target").count().toPandas()

,target,count
0,1,34214
1,2,268492
2,0,324114


In [26]:
# 1. Combine counts into one dataframe
df_all = pd.concat([
    df_target.groupBy("target").count().toPandas().assign(Stage="Before"),
    df_downsampled.groupBy("target").count().toPandas().assign(Stage="After")
])

# 2. Plot side-by-side charts using facet_col
px.bar(df_all, x="target", y="count", facet_col="Stage", width=700, height=350)

# 5. Machine Learning Feature Pipeline & Data Normalization

---

## 5.1 PySpark Feature Vector Assembly (`VectorAssembler`)
We consolidate all processed numerical and one-hot encoded categorical variables into a single feature vector (`features_to_scale`).

* **Feature Vector Dimension**: **86 distinct predictor features**.

In [28]:
# 1. Identify feature columns
feature_cols = [c for c in df_downsampled.columns if c not in ["id", "target"]]

# 2. Cast non-vector feature columns to double to prevent type errors
df_downsampled = df_downsampled.select([
    col(c).cast("double") if not c.endswith("_vec") else col(c) 
    for c in df_downsampled.columns
])

# 3. Assemble features (86 features guaranteed)
df_downsampled = VectorAssembler(inputCols=feature_cols, outputCol="features_to_scale").transform(df_downsampled)
df_downsampled.select("features_to_scale", "target").show(5)

+--------------------+------+
|   features_to_scale|target|
+--------------------+------+
|(86,[0,2,3,4,5,6,...|   0.0|
|(86,[0,2,3,4,5,6,...|   0.0|
|(86,[0,2,3,4,5,6,...|   2.0|
|(86,[0,2,3,4,5,6,...|   0.0|
|(86,[0,2,3,4,5,6,...|   0.0|
+--------------------+------+
only showing top 5 rows


In [29]:
print(f"Total columns: {len(feature_cols)}\nColumns: {feature_cols}")

Total columns: 21
Columns: ['term', 'acc_now_delinq', 'annual_inc', 'application_type', 'dti', 'installment', 'int_rate', 'loan_amnt', 'tax_liens', 'delinq_amnt', 'policy_code', 'last_fico_range_high', 'last_fico_range_low', 'recoveries', 'collection_recovery_fee', 'verification_status_encoded', 'grade_index', 'purpose_vec', 'addr_state_vec', 'home_ownership_vec', 'initial_list_status_vec']


---

## 5.2 Train / Validation / Test Partitioning & Feature Normalization (`MinMaxScaler`)

### ✂️ Data Splitting:
* **Train Split (80%)**: Parameter optimization.
* **Validation Split (10%)**: Hyperparameter tuning and model selection.
* **Test Split (10%)**: Holdout evaluation for unbiased performance measurement.

### 📏 Feature Scaling Rationale:
`MinMaxScaler` is fit **strictly on the Training split** and applied across all splits to map values to $[0, 1]$, preventing data leakage while accelerating neural network convergence.

In [30]:
# 1. Split data (80% Train, 10% Test, 10% Validation)
train_data, temp_data = df_downsampled.randomSplit([0.8, 0.2], seed=42)
test_data, val_data = temp_data.randomSplit([0.5, 0.5], seed=42)

# 2. Fit MinMaxScaler on train set and transform all splits directly
scaler = MinMaxScaler(inputCol="features_to_scale", outputCol="features").fit(train_data)
train_data, val_data, test_data = scaler.transform(train_data), scaler.transform(val_data), scaler.transform(test_data)

# 6. Model Training, Performance Benchmarking & Evaluation

---

## 6.1 Multi-Model Benchmarking Suite
We evaluate three model architectures tailored for enterprise credit underwriting:

1. **Logistic Regression**: Baseline linear model offering high interpretability required for regulatory credit scoring scorecards.
2. **Random Forest Classifier**: Ensemble tree model capable of capturing complex non-linear feature interactions.
3. **Multilayer Perceptron (MLP) Neural Network**: Deep architecture (`[86, 64, 32, 3]`) for feature representation across high-dimensional credit data.

In [ ]:
# 1. Pre-load data splits directly into RAM
for ds in [train_data, val_data, test_data]: ds.cache().count()

# 2. Shared Evaluator instance
evaluator = MulticlassClassificationEvaluator(labelCol="target")

# 3. Model training & evaluation loop
def evaluate(model, name):
    fit_m = model.fit(train_data)
    res = {"Model": name}
    for label, ds in [("Train", train_data), ("Validation", val_data), ("Test", test_data)]:
        preds = fit_m.transform(ds)  # Transform once per split
        res[f"Accuracy ({label})"] = round(evaluator.setMetricName("accuracy").evaluate(preds), 3)
        res[f"F1 Score ({label})"] = round(evaluator.setMetricName("f1").evaluate(preds), 3)
    return res

models = [
    (LogisticRegression(featuresCol="features", labelCol="target"), "LogisticRegression"),
    (RandomForestClassifier(featuresCol="features", labelCol="target"), "Random Forest"),
    (MultilayerPerceptronClassifier(layers=[86, 64, 32, 3], featuresCol="features", labelCol="target", maxIter=20, seed=123), "Neural Network")
]

pd.DataFrame([evaluate(m, name) for m, name in models])

,Model,Accuracy (Train),F1 Score (Train),Accuracy (Validation),F1 Score (Validation),Accuracy (Test),F1 Score (Test)
0,LogisticRegression,0.879,0.861,0.881,0.864,0.877,0.859
1,Random Forest,0.866,0.842,0.869,0.845,0.865,0.840
2,Neural Network,0.752,0.730,0.752,0.731,0.750,0.728


---

## 7. Executive Summary & Strategic Business Recommendations

### 📊 Model Performance & Architectural Trade-offs:
* **Logistic Regression**: Offers full model transparency and scoreable odds ratios for regulatory submission under FCRA (Fair Credit Reporting Act) and ECOA (Equal Credit Opportunity Act) compliance.
* **Random Forest**: Efficiently captures risk non-linearities (e.g., compounding risk of high DTI combined with low FICO scores) without requiring manual interaction engineering.
* **Multilayer Perceptron (MLP)**: Effective at high-dimensional feature abstraction; recommended for automated shadow-scoring parallel pipelines.

---

### 🎯 Key Executive Recommendations:

1. **Dynamic Risk-Based Interest Rate Pricing**:
   - Integrate predicted default probabilities ($P(\text{Default})$) directly into automated pricing engines to dynamically set interest rate premiums that offset expected credit loss.

2. **Early Warning Loss Mitigation System (EWS)**:
   - Deploy Class 1 (`Late / Grace Period`) alerts to trigger proactive risk mitigation, loan restructuring, or secondary market portfolio sales prior to full charge-off.

3. **Risk-Adjusted Return on Capital (RAROC) Threshold Tuning**:
   - Align decision thresholds with loss severity matrices: avoiding a single default saves 100% principal loss, whereas misclassifying a performing borrower forfeits only marginal interest income.

---

### 📌 Executive Briefing Notes:
> *"This analytics initiative delivers a scalable, enterprise-grade PySpark credit default forecasting pipeline across 2.2 million LendingClub loan records. By addressing severe class imbalance through controlled downsampling and benchmarking interpretable linear scorecards against ensemble and neural network architectures, the platform provides financial institutions with a robust tool to reduce default write-offs, optimize interest rate pricing, and protect institutional capital."*